In [ ]:
"""
Solar Simulator — Python Controller
Drives pan/tilt stepper motors through a simulator Arduino Mega,
which also forwards sun position data to a tower heliostat.
"""

import time
import math
import serial
import threading
import pvlib
import pandas as pd


# CONFIGURATION


LAT, LON = 42.3601, -71.0589
SIM_TZ   = 'America/New_York'

# Serial
SERIAL_PORT    = 'COM4'
BAUD_RATE      = 115200
SERIAL_TIMEOUT = 10
SETTLE_TIME    = 1.0

# Playback
TARGET_INTERVAL = 0.8
MAX_WAYPOINTS   = 2000
MAX_PLAYBACK    = 28800

# Pan axis (belt drive, DM542)
PAN_STEPS_PER_REV = 200
PAN_MICROSTEPPING = 4
PAN_GEAR_RATIO    = 5
PAN_BELT_CIRC_MM  = 1219.2
PAN_PULLEY_CIRC_MM = 16 * 2.0  # teeth × pitch
PAN_STEPS_REV      = PAN_STEPS_PER_REV * PAN_MICROSTEPPING * PAN_GEAR_RATIO
PAN_STEPS_PER_DEG  = (PAN_BELT_CIRC_MM / PAN_PULLEY_CIRC_MM * PAN_STEPS_REV) / 360.0

# Tilt axis (rack & pinion, TB6600)
TILT_STEPS_PER_REV = 200
TILT_MICROSTEPPING = 4
TILT_GEAR_RATIO    = 20
TILT_STEPS_REV     = TILT_STEPS_PER_REV * TILT_MICROSTEPPING * TILT_GEAR_RATIO
TILT_STEPS_PER_DEG = (TILT_STEPS_REV * math.pi * 657.5) / (math.pi * 100.0 * 180.0)

# Drift check — every N waypoints during tracking
DRIFT_CHECK_INTERVAL = 10   # check every 10th waypoint
DRIFT_TOLERANCE      = 20

# Threading
_serial_lock = threading.Lock()


# SERIAL HELPERS


def _send(ser, cmd, timeout=None):
    """Send command, return response. Thread-safe."""
    with _serial_lock:
        ser.reset_input_buffer()
        if timeout:
            old, ser.timeout = ser.timeout, timeout
        ser.write(cmd.encode() if isinstance(cmd, str) else cmd)
        ser.flush()
        resp = ser.readline().decode('utf-8', errors='ignore').strip()
        if timeout:
            ser.timeout = old
    return resp


def send_target(ser, pan, tilt, interval_ms, az=0, el=0):
    """Non-blocking tracking move with sun position forwarding."""
    resp = _send(ser, f"T{pan},{tilt},{interval_ms:.0f},{az:.4f},{el:.4f}\n")
    if resp != "TOK":
        print(f"  [WARN] T command: '{resp}'", flush=True)


def send_blocking_move(ser, axis, steps):
    """Blocking relative move (P=pan, T=tilt)."""
    print(f"  → M{axis}{steps}", end="", flush=True)
    resp = _send(ser, f"M{axis}{steps}\n")
    print("  ✓" if resp == "OK" else f"  [WARN] '{resp}'", flush=True)


def get_status(ser):
    """Returns (pan_pos, tilt_pos, pan_ok, tilt_ok) or None."""
    line = _send(ser, b"S\n")
    if not line.startswith("S:"):
        return None
    parts = line[2:].split(",")
    if len(parts) < 4:
        return None
    try:
        return int(parts[0]), int(parts[1]), parts[2] == "1", parts[3] == "1"
    except ValueError:
        return None


def verify_position(ser, exp_pan, exp_tilt, tol=10):
    """Check motor position against expected values."""
    status = get_status(ser)
    if status is None:
        print("  [WARN] Could not read status", flush=True)
        return False
    pan, tilt, _, _ = status
    pan_ok  = abs(pan  - exp_pan)  <= tol
    tilt_ok = abs(tilt - exp_tilt) <= tol
    print(f"  [VERIFY] pan={pan/PAN_STEPS_PER_DEG:6.2f}°({pan:>7}){'✓' if pan_ok else '⚠'}  "
          f"tilt={tilt/TILT_STEPS_PER_DEG:5.2f}°({tilt:>7}){'✓' if tilt_ok else '⚠'}", flush=True)
    return pan_ok and tilt_ok


def settle_and_verify(ser, exp_pan, exp_tilt):
    """Wait, flush, verify — used after every blocking move."""
    time.sleep(SETTLE_TIME)
    ser.reset_input_buffer()
    return verify_position(ser, exp_pan, exp_tilt)


def wake_tower(ser):
    """Send START to tower via simulator. Returns immediately."""
    print("  Sending START to tower...", flush=True)
    resp = _send(ser, b"W\n")
    ok = resp == "WOK"
    print(f"  {'START sent successfully.' if ok else f'Unexpected: {resp!r}'}", flush=True)
    return ok


# DRIFT CHECK (inline, called from tracking loop)


def check_drift(ser, exp_pan, exp_tilt):
    """Quick drift check — returns True if within tolerance."""
    status = get_status(ser)
    if status is None:
        return True  # can't check, assume ok
    pan, tilt, _, _ = status
    pd_ = abs(pan - exp_pan)
    td  = abs(tilt - exp_tilt)
    if pd_ > DRIFT_TOLERANCE or td > DRIFT_TOLERANCE:
        msg = "  [DRIFT]"
        if pd_ > DRIFT_TOLERANCE: msg += f" pan off by {pd_} steps"
        if td  > DRIFT_TOLERANCE: msg += f" tilt off by {td} steps"
        print(msg, flush=True)
        return False
    return True


# SUN PATH


def get_daylight_window(date_str):
    times = pd.date_range(
        pd.Timestamp(f"{date_str} 00:00", tz=SIM_TZ),
        pd.Timestamp(f"{date_str} 23:59", tz=SIM_TZ),
        freq="1min"
    )
    solpos = pvlib.solarposition.get_solarposition(times, LAT, LON)
    day = solpos[solpos['elevation'] > 0]
    if day.empty:
        return None, None
    return day.index[0].strftime("%H:%M"), day.index[-1].strftime("%H:%M")


def precompute_path(date_str, start, end, n_points):
    times  = pd.date_range(
        pd.Timestamp(f"{date_str} {start}", tz=SIM_TZ),
        pd.Timestamp(f"{date_str} {end}",   tz=SIM_TZ),
        periods=n_points
    )
    solpos = pvlib.solarposition.get_solarposition(times, LAT, LON)
    return [(ts, row['azimuth'], 90.0 - row['zenith'])
            for ts, row in solpos.iterrows()
            if 90.0 - row['zenith'] >= 0]



# USER INPUT


def get_user_inputs():
    print("=" * 55)
    print("  SOLAR SIMULATOR CONFIGURATION")
    print("=" * 55 + "\n", flush=True)

    while True:
        raw = input("  Date (YYYY-MM-DD): ").strip()
        try:
            pd.Timestamp(raw)
            break
        except Exception:
            print("    Invalid format.")
    sim_date = raw

    print("  Computing daylight window...", flush=True)
    sunrise, sunset = get_daylight_window(sim_date)
    if sunrise is None:
        print("  No daylight. Exiting.")
        return None, None, None, None
    print(f"  Sunrise: {sunrise}   Sunset: {sunset}", flush=True)

    while True:
        raw = input(f"  Playback duration in seconds (max {MAX_PLAYBACK}): ").strip()
        try:
            val = int(raw)
            if 0 < val <= MAX_PLAYBACK:
                break
        except ValueError:
            pass
        print(f"    Enter 1–{MAX_PLAYBACK}.")

    print(flush=True)
    return sim_date, sunrise, sunset, val


# MAIN


def main():
    sim_date, sunrise, sunset, duration = get_user_inputs()
    if sim_date is None:
        return

    n_waypoints = min(MAX_WAYPOINTS, max(100, round(duration / TARGET_INTERVAL)))

    print("=" * 55)
    print("  SOLAR TRACKER")
    print("=" * 55)
    print(f"  Pan:  {PAN_STEPS_PER_DEG:.2f} steps/deg  (belt, DM542)")
    print(f"  Tilt: {TILT_STEPS_PER_DEG:.2f} steps/deg  (rack & pinion, TB6600)\n", flush=True)

    # Pre-compute sun path
    print(f"  Computing sun path for {sim_date} ({sunrise} → {sunset})...", flush=True)
    path = precompute_path(sim_date, sunrise, sunset, n_waypoints)
    if not path:
        print("  No waypoints. Exiting.")
        return

    interval = duration / len(path)
    print(f"  {len(path)} waypoints  ({interval*1000:.0f}ms interval)")
    print(f"  First: az={path[0][1]:.1f}°  el={path[0][2]:.1f}°")
    print(f"  Last:  az={path[-1][1]:.1f}°  el={path[-1][2]:.1f}°\n", flush=True)

    # Open serial
    print(f"Opening {SERIAL_PORT} at {BAUD_RATE} baud...", flush=True)
    with serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=SERIAL_TIMEOUT) as ser:
        time.sleep(2)
        ser.reset_input_buffer()
        print("  Connected.\n", flush=True)

        # Step 1: Tower
        print("=" * 55)
        print("  STEP 1 — TOWER HOMING")
        print("=" * 55)
        tower_ok = False
        if input("  Home tower? (y/n): ").strip().lower() == 'y':
            print("\n  Reset the tower Arduino NOW (press the button),", flush=True)
            input("  then press ENTER once it's powered back up... ")
            time.sleep(2)
            if wake_tower(ser):
                print("\n  Tower is homing. Watch its Serial Monitor (9600 baud).", flush=True)
                input("  Press ENTER when homing is complete... ")
                tower_ok = True
                print("  Tower ready.\n", flush=True)
            else:
                print("  Could not send START.\n", flush=True)
        else:
            print("  Tower skipped.\n", flush=True)

        # Step 2: Simulation
        first_az, first_el = path[0][1], path[0][2]
        first_pan  = round(first_az * PAN_STEPS_PER_DEG)
        first_tilt = round(first_el * TILT_STEPS_PER_DEG)

        print("=" * 55)
        print("  STEP 2 — SIMULATION")
        print("=" * 55)
        print(f"  {len(path)} waypoints over {duration}s")
        print(f"  First: az={first_az:.1f}°  el={first_el:.1f}°")
        print(f"  Tower: {'ACTIVE' if tower_ok else 'OFFLINE'}")
        input("  Press ENTER to begin... ")
        print(flush=True)

        # Slew to sunrise
        print(f"  Slewing to sunrise ({first_az:.1f}°az / {first_el:.1f}°el)...", flush=True)
        send_blocking_move(ser, 'P', first_pan)
        time.sleep(SETTLE_TIME)
        ser.reset_input_buffer()
        send_blocking_move(ser, 'T', first_tilt)
        settle_and_verify(ser, first_pan, first_tilt)

        # Send first position to tower so it can slew to its bisect angle
        if tower_ok:
            print(f"  Sending initial position to tower...", flush=True)
            send_target(ser, first_pan, first_tilt, 5000, first_az, first_el)
            input("  Press ENTER when tower has finished slewing... ")
            print(flush=True)

        cur_pan, cur_tilt = first_pan, first_tilt

        # Tracking
        print("=" * 55)
        print("  TRACKING")
        print("=" * 55)
        print(f"  {len(path)} waypoints  ({interval*1000:.0f}ms interval)")
        print("  Press ENTER to stop.\n", flush=True)

        stop_event = threading.Event()
        threading.Thread(target=lambda: (input(), stop_event.set()), daemon=True).start()

        t0 = time.time()

        for i, (ts, az, el) in enumerate(path):
            if stop_event.is_set():
                print(f"\n\n  Stopped at waypoint {i}/{len(path)}.", flush=True)
                break

            tgt_pan  = round(az * PAN_STEPS_PER_DEG)
            tgt_tilt = round(el * TILT_STEPS_PER_DEG)

            wait = (t0 + i * interval) - time.time()
            if wait > 0:
                time.sleep(wait)

            send_target(ser, tgt_pan, tgt_tilt, interval * 1000, az, el)
            cur_pan, cur_tilt = tgt_pan, tgt_tilt

            progress = (i + 1) / len(path) * 100
            line = (f"  [{ts.strftime('%H:%M:%S')}] ({progress:5.1f}%) "
                    f"az={az:6.2f}°  el={el:5.2f}°  "
                    f"pan={tgt_pan:>7}  tilt={tgt_tilt:>7}")

            # Inline drift check every N waypoints
            if (i + 1) % DRIFT_CHECK_INTERVAL == 0:
                if check_drift(ser, tgt_pan, tgt_tilt):
                    line += "  ✓"

            print(line, flush=True)

        # Summary
        elapsed = time.time() - t0
        print(f"\n  {'Stopped early' if stop_event.is_set() else 'Simulation complete'}  "
              f"({elapsed:.1f}s)", flush=True)
        settle_and_verify(ser, cur_pan, cur_tilt)

        # Return home
        print(f"\n  Returning to home...", flush=True)
        time.sleep(SETTLE_TIME)
        ser.reset_input_buffer()
        send_blocking_move(ser, 'T', -cur_tilt)
        time.sleep(SETTLE_TIME)
        ser.reset_input_buffer()
        send_blocking_move(ser, 'P', -cur_pan)
        settle_and_verify(ser, 0, 0)
        print("  Home. Done.", flush=True)

if __name__ == "__main__":
    main()